<a href="https://colab.research.google.com/github/georgiavining/speed-angle-prediction-CNN/blob/main/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#mounting drive
from google.colab import drive
drive.mount('/content/drive')

#creating paths
import sys
sys.path.append('/content/drive/MyDrive/mlis_p2/kaggle/source')
import os
DATA_PATH = "/content/drive/MyDrive/mlis_p2/kaggle/data"
TRAIN_CSV = os.path.join(DATA_PATH, "train.csv")
OUTPUTS_PATH = "/content/drive/MyDrive/mlis_p2/kaggle/outputs"
PREDICTIONS_PATH = os.path.join(OUTPUTS_PATH, "predictions")
IMAGES_PATH = "/content/drive/MyDrive/mlis_p2/kaggle/notebooks/images"

TRAIN_IMAGES = '/content/training_images'
TEST_IMAGES = '/content/test_images'

# copying images from drive to local disk
if not os.path.exists(TRAIN_IMAGES):
    print("Copying training images to local disk...")
    !cp -r /content/drive/MyDrive/mlis_p2/kaggle/data/training_images /content/training_images
    print("Done!")

if not os.path.exists(TEST_IMAGES):
    print("Copying test images to local disk...")
    !cp -r /content/drive/MyDrive/mlis_p2/kaggle/data/test_images /content/test_images
    print("Done!")

Mounted at /content/drive
Copying training images to local disk...
^C
Done!
Copying test images to local disk...
^C
Done!


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
%tensorflow_version 2.x
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import pandas as pd

Colab only includes TensorFlow 2.x; %tensorflow_version has no effect.


# Preparing Datasets

In [ ]:
df = pd.read_csv(TRAIN_CSV)

#checking invalid images
invalid_images = []
for idx in df['image_id']:
    img_path = os.path.join(TRAIN_IMAGES, f"{idx}.png")
    if not os.path.exists(img_path) or os.path.getsize(img_path) == 0:
        invalid_images.append(idx)

#removing invalid images from df
df = df[~df['image_id'].isin(invalid_images)]
print("Number of valid training samples:", len(df))

Number of valid training samples: 14383


In [ ]:
def preprocess_image(filename, img_size):
    img = tf.io.read_file(filename)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [img_size, img_size])
    img = tf.image.convert_image_dtype(img, tf.float32)
    return img

def build_dataset(filenames, labels=None, img_size=224, batch_size=32,
                  shuffle=False, shuffle_buffer=1000, seed=42):
    """
    Single function for all dataset types:
    - labels=None         → test dataset (no labels)
    - labels=array        → single output dataset (angle or speed)
    - labels=(arr1, arr2) → dual output dataset (angle + speed dict)
    """
    if labels is None:
        ds = tf.data.Dataset.from_tensor_slices(filenames)
        ds = ds.map(lambda f: preprocess_image(f, img_size), num_parallel_calls=2)
    elif isinstance(labels, tuple):
        ds = tf.data.Dataset.from_tensor_slices((filenames, labels[0], labels[1]))
        ds = ds.map(lambda f, a, s: (preprocess_image(f, img_size), {"angle": a, "speed": s}),
                    num_parallel_calls=2)
    else:
        ds = tf.data.Dataset.from_tensor_slices((filenames, labels))
        ds = ds.map(lambda f, l: (preprocess_image(f, img_size), l), num_parallel_calls=2)

    if shuffle:
        ds = ds.shuffle(shuffle_buffer, seed=seed)

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

def split_indices(n_total, val_split=0.1, test_split=0.1, seed=42):
    np.random.seed(seed)
    idx = np.random.permutation(n_total)
    n_test = int(n_total * test_split)
    n_val = int(n_total * val_split)
    n_train = n_total - n_val - n_test
    return idx[:n_train], idx[n_train:n_train+n_val], idx[n_train+n_val:]

In [ ]:
filenames = [os.path.join(TRAIN_IMAGES, f"{i}.png") for i in df['image_id']]
angles = df['angle'].values
speeds = df['speed'].values

train_idx, val_idx, test_idx = split_indices(len(filenames))
fn = np.array(filenames)

# Angle datasets
angle_train = build_dataset(fn[train_idx], angles[train_idx], shuffle=True)
angle_val   = build_dataset(fn[val_idx],   angles[val_idx])
angle_test  = build_dataset(fn[test_idx],  angles[test_idx])

# Speed datasets
speed_train = build_dataset(fn[train_idx], speeds[train_idx], shuffle=True)
speed_val   = build_dataset(fn[val_idx],   speeds[val_idx])
speed_test  = build_dataset(fn[test_idx],  speeds[test_idx])

# Building Models

In [ ]:
def build_angle_model(
    input_shape=(224, 224, 3),
    conv_filters=[64, 128, 256, 512],
    dense_units=[256, 128],
    dropout=0.3
):
    inputs = tf.keras.layers.Input(shape=input_shape)
    x = inputs

    for i, filters in enumerate(conv_filters):
        x = tf.keras.layers.Conv2D(filters, (3,3), padding='same', name=f'conv{i+1}')(x)
        x = tf.keras.layers.BatchNormalization(name=f'bn{i+1}')(x)
        x = tf.keras.layers.Activation('relu', name=f'act{i+1}')(x)
        x = tf.keras.layers.MaxPooling2D((2,2), name=f'pool{i+1}')(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    for i, units in enumerate(dense_units):
        x = tf.keras.layers.Dense(units, activation='relu', name=f'dense{i+1}')(x)
        x = tf.keras.layers.Dropout(dropout, name=f'dropout{i+1}')(x)

    # Linear output — angle is continuous regression
    angle_out = tf.keras.layers.Dense(1, activation='sigmoid', name='angle')(x)

    return tf.keras.models.Model(inputs=inputs, outputs=angle_out)

def build_speed_model(
    input_shape=(224, 224, 3),
    conv_filters=[32, 64, 128, 256],  # lighter — speed is simpler
    dense_units=[128],
    dropout=0.5  # higher dropout — binary task, prone to overfitting
):
    inputs = tf.keras.layers.Input(shape=input_shape)
    x = inputs

    for i, filters in enumerate(conv_filters):
        x = tf.keras.layers.Conv2D(filters, (3,3), padding='same', name=f'conv{i+1}')(x)
        x = tf.keras.layers.BatchNormalization(name=f'bn{i+1}')(x)
        x = tf.keras.layers.Activation('relu', name=f'act{i+1}')(x)
        x = tf.keras.layers.MaxPooling2D((2,2), name=f'pool{i+1}')(x)

    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    for i, units in enumerate(dense_units):
        x = tf.keras.layers.Dense(units, activation='relu', name=f'dense{i+1}')(x)
        x = tf.keras.layers.Dropout(dropout, name=f'dropout{i+1}')(x)

    # Sigmoid output — speed is binary
    speed_out = tf.keras.layers.Dense(1, activation='sigmoid', name='speed')(x)

    return tf.keras.models.Model(inputs=inputs, outputs=speed_out)

# Training Models

In [ ]:
#adding class weights to adjust for class imbalance
total = 11439 + 2944
weight_for_0 = total / (2 * 2944)   # ~2.4
weight_for_1 = total / (2 * 11439)  # ~0.6

def weighted_bce(y_true, y_pred):
    weights = y_true * weight_for_1 + (1 - y_true) * weight_for_0
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(weights * bce)

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1 (Conv2D)      │ (None, 224, 224,  │        896 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 112, 112,  │          0 │ conv1[0][0]       │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2 (Conv2D)      │ (None, 112, 112,  │     18,496 │ pool1[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool2               │ (None, 56, 56,    │          0 │ conv2[0][0]       │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv3 (Conv2D)      │ (None, 56, 56,    │     73,856 │ pool2[0][0]       │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool3               │ (None, 28, 28,    │          0 │ conv3[0][0]       │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 100352)    │          0 │ pool3[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense1 (Dense)      │ (None, 128)       │ 12,845,184 │ flatten[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ angle (Dense)       │ (None, 1)         │        129 │ dense1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ speed (Dense)       │ (None, 1)         │        129 │ dense1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 12,938,690 (49.36 MB)

 Trainable params: 12,938,690 (49.36 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
tf.keras.backend.clear_session()
lr = 1e-3

angle_model = build_angle_model()
angle_model.compile(
    loss='mse',
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
    metrics=['mse']
)

speed_model = build_speed_model()
speed_model.compile(
    loss=weighted_bce,
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
    metrics=['mse']
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=7, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]

angle_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath= os.path.join(MODELS_PATH, 'angle_model_best.keras'),    #<----  change
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

history_angle = angle_model.fit(
    angle_train,
    epochs=50,
    validation_data=angle_val,
    callbacks=callbacks+[angle_checkpoint],
    verbose=1
)

speed_checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath= os.path.join(MODELS_PATH, 'speed_model_best.keras'),       #<----  change
    monitor='val_loss',
    save_best_only=True,
    verbose=1
)

history_speed = speed_model.fit(
    speed_train,
    epochs=100,
    validation_data=speed_val,
    callbacks=callbacks+[speed_checkpoint],
    verbose=1
)

In [ ]:
def plot_metric(history, metric,save_dir, model_name):
    plt.figure()

    plt.plot(history.history[metric], label=f"train {metric}")
    plt.plot(history.history[f"val_{metric}"], label=f"val {metric}")

    plt.xlabel("Epoch")
    plt.ylabel(metric)
    plt.title(f"{metric}")
    plt.legend()
    plt.grid(alpha=0.3)

    filename = f"{model_name}_{metric}.png"
    plt.savefig(os.path.join(save_dir, filename),
                dpi=300, bbox_inches="tight")

    plt.show()
    plt.close()

plot_metric(history_angle, "mse", IMAGES_PATH, "model6_angle")      #<----  change
plot_metric(history_speed, "mse", IMAGES_PATH, "model6_speed")      #<----  change

NameError: name 'plot_metric' is not defined

# Test Set Evaluation

In [ ]:
#angle_model = tf.keras.models.load_model(os.path.join(MODELS_PATH, 'angle_model_best.keras'))
#speed_model = tf.keras.models.load_model(os.path.join(MODELS_PATH, 'speed_model_best.keras'), custom_objects={'weighted_bce': weighted_bce})

In [ ]:
print("--- Angle Model ---")
angle_results = angle_model.evaluate(angle_test, verbose=1)

print("--- Speed Model ---")
speed_results = speed_model.evaluate(speed_test, verbose=1)

angle_mse = angle_results[1]
speed_mse = speed_results[1]

print(f"\nAngle MSE:    {angle_mse:.5f}")
print(f"Speed MSE:    {speed_mse:.5f}")
print(f"Combined MSE: {((angle_mse + speed_mse) / 2):.5f}")

NameError: name 'final_model' is not defined

# Making Predictions

In [ ]:
def generate_test_predictions(angle_model, speed_model, test_ds,
                                        predictions_path, model_name):

    angle_preds, speed_preds = [], []

    for x in test_ds:
        angle_preds.extend(angle_model.predict(x, verbose=0).flatten())
        speed_preds.extend((speed_model.predict(x, verbose=0) > 0.5).astype(int).flatten())

    df_preds = pd.DataFrame({
        "image_id": range(len(angle_preds)),
        "angle": np.array(angle_preds),
        "speed": np.array(speed_preds)
    })

    os.makedirs(predictions_path, exist_ok=True)
    csv_output = os.path.join(predictions_path, f"{model_name}.csv")
    df_preds.to_csv(csv_output, index=False)
    print(f"Saved to: {csv_output}")
    return df_preds

In [ ]:
test_filenames = sorted(
    [os.path.join(TEST_IMAGES, f) for f in os.listdir(TEST_IMAGES) if f.endswith(".png")],
    key=lambda x: int(os.path.splitext(os.path.basename(x))[0])
)

test_ds = build_dataset(test_filenames)

generate_test_predictions(
    angle_model=angle_model,
    speed_model=speed_model,
    test_ds=test_ds,
    predictions_path=PREDICTIONS_PATH,
    model_name="model_6"                    #<----  change
)